# EDA Music Mood Recommender

Расширенный разведочный анализ датасета для финального ML-проекта. Ноутбук показывает структуру данных, пропуски, распределения музыкальных признаков, жанры, корреляции и классы настроения.

## 1. Импорт библиотек и загрузка данных

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid", palette="Set2")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12

DATA_PATH = "dataset.csv"
df_raw = pd.read_csv(DATA_PATH)
df_raw.head()

## 2. Общая информация о датасете

In [ ]:
print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]:,}")
display(df_raw.dtypes.to_frame("dtype"))
display(df_raw.describe(include="all").T.head(30))

## 3. Пропуски и дубликаты

In [ ]:
missing = df_raw.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_table = pd.DataFrame({"missing_count": missing, "missing_percent": missing_pct})
display(missing_table[missing_table["missing_count"] > 0])
print("Duplicate track_id:", df_raw.duplicated("track_id").sum() if "track_id" in df_raw.columns else "no track_id")
print("Full duplicated rows:", df_raw.duplicated().sum())

## 4. Очистка данных и создание mood

In [ ]:
df = df_raw.copy()
unnamed = [c for c in df.columns if c.startswith("Unnamed")]
df = df.drop(columns=unnamed)
required = ["track_id", "artists", "album_name", "track_name", "track_genre", "explicit", "popularity", "duration_ms", "danceability", "energy", "key", "loudness", "mode", "speechiness", "acousticness", "instrumentalness", "liveness", "valence", "tempo", "time_signature"]
df = df.dropna(subset=required).drop_duplicates(subset=["track_id"]).reset_index(drop=True)
df["artist_primary"] = df["artists"].astype(str).str.split(";").str[0]

def assign_mood(row):
    energy = row["energy"]
    valence = row["valence"]
    danceability = row["danceability"]
    acousticness = row["acousticness"]
    if danceability >= 0.72 and energy >= 0.62:
        return "dance"
    if acousticness >= 0.65 and energy <= 0.50:
        return "chill"
    if energy >= 0.60 and valence >= 0.60:
        return "happy"
    if energy >= 0.60 and valence <= 0.40:
        return "intense"
    if energy <= 0.45 and valence >= 0.55:
        return "calm"
    if energy <= 0.45 and valence <= 0.45:
        return "sad"
    return "neutral"

df["mood"] = df.apply(assign_mood, axis=1)
print(f"Rows after cleaning: {len(df):,}")
display(df[["track_name", "artists", "track_genre", "energy", "valence", "danceability", "acousticness", "mood"]].head())

## 5. Исключение признаков, использованных для создания mood

Чтобы классификатор не повторял правила напрямую, признаки `energy`, `valence`, `danceability`, `acousticness` используются только для создания pseudo-label `mood`, но исключаются из признаков обучения модели. Рекомендательные алгоритмы всё равно используют полный audio-вектор, потому что для content-based recommendations это корректно.

In [ ]:
target_derived_features = ["danceability", "energy", "valence", "acousticness"]
all_numeric_features = ["popularity", "duration_ms", "danceability", "energy", "key", "loudness", "mode", "speechiness", "acousticness", "instrumentalness", "liveness", "valence", "tempo", "time_signature"]
classifier_numeric_features = [feature for feature in all_numeric_features if feature not in target_derived_features]
pd.DataFrame({
    "target_created_from": pd.Series(target_derived_features),
    "classifier_uses_numeric": pd.Series(classifier_numeric_features)
})

**Интерпретация:** такая постановка снижает leakage-like эффект. Accuracy становится ниже, зато модель решает более честную задачу: пытается восстановить настроение по косвенным признакам (`tempo`, `loudness`, `speechiness`, `instrumentalness`, `genre` и т.д.).

## 6. Распределение жанров

In [ ]:
top_genres = df["track_genre"].value_counts().head(20)
plt.figure(figsize=(13, 7))
sns.barplot(x=top_genres.values, y=top_genres.index, palette="viridis")
plt.title("Top 20 genres by track count")
plt.xlabel("Tracks")
plt.ylabel("Genre")
plt.show()

## 7. Распределение классов настроения

In [ ]:
mood_counts = df["mood"].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=mood_counts.index, y=mood_counts.values, ax=axes[0], palette="Set2")
axes[0].set_title("Mood class distribution")
axes[0].set_xlabel("Mood")
axes[0].set_ylabel("Tracks")
axes[1].pie(mood_counts.values, labels=mood_counts.index, autopct="%1.1f%%", startangle=90)
axes[1].set_title("Mood class share")
plt.tight_layout()
plt.show()

## 8. Распределения основных audio features

In [ ]:
audio_features = ["danceability", "energy", "valence", "acousticness", "instrumentalness", "speechiness", "liveness", "tempo", "popularity", "loudness"]
df[audio_features].hist(bins=40, figsize=(16, 12), color="#1DB954", edgecolor="#121212")
plt.suptitle("Audio feature distributions", fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

## 9. Корреляции признаков

In [ ]:
corr = df[audio_features].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Greens", center=0, linewidths=.5)
plt.title("Correlation heatmap of audio features")
plt.show()
display(corr.unstack().sort_values(ascending=False).drop_duplicates().head(15).to_frame("correlation"))

## 10. Energy vs Valence mood map

In [ ]:
sample = df.sample(min(12000, len(df)), random_state=42)
plt.figure(figsize=(11, 8))
sns.scatterplot(data=sample, x="valence", y="energy", hue="mood", alpha=0.55, s=18, palette="tab10")
plt.axhline(0.5, color="black", linestyle="--", linewidth=1, alpha=.35)
plt.axvline(0.5, color="black", linestyle="--", linewidth=1, alpha=.35)
plt.title("Mood map: energy vs valence")
plt.xlabel("Valence: sad -> happy")
plt.ylabel("Energy: calm -> energetic")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 11. Признаки по mood-классам

In [ ]:
mood_profile = df.groupby("mood")[audio_features].mean().round(3)
display(mood_profile)
plt.figure(figsize=(12, 6))
sns.heatmap(mood_profile[["danceability", "energy", "valence", "acousticness", "instrumentalness", "speechiness", "popularity"]], annot=True, cmap="YlGnBu", vmin=0, vmax=1)
plt.title("Average audio features by mood")
plt.show()

## 12. Популярность по жанрам и настроениям

In [ ]:
genre_popularity = df.groupby("track_genre")["popularity"].mean().sort_values(ascending=False).head(20)
plt.figure(figsize=(13, 7))
sns.barplot(x=genre_popularity.values, y=genre_popularity.index, palette="mako")
plt.title("Top genres by average popularity")
plt.xlabel("Average popularity")
plt.ylabel("Genre")
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="mood", y="popularity", palette="Set2")
plt.title("Popularity distribution by mood")
plt.show()

## 13. Артисты: агрегированные признаки

In [ ]:
artist_profile = df.groupby("artist_primary").agg(
    tracks_count=("track_id", "count"),
    mean_popularity=("popularity", "mean"),
    mean_energy=("energy", "mean"),
    mean_danceability=("danceability", "mean"),
    mean_valence=("valence", "mean"),
    top_genre=("track_genre", lambda s: s.mode().iat[0] if not s.mode().empty else "unknown")
).query("tracks_count >= 2").sort_values("mean_popularity", ascending=False)
display(artist_profile.head(20))
plt.figure(figsize=(10, 6))
sns.scatterplot(data=artist_profile.head(1000), x="mean_energy", y="mean_valence", size="tracks_count", hue="mean_popularity", palette="viridis", alpha=.65)
plt.title("Artists by average energy, valence and popularity")
plt.show()

## 14. Выводы для модели и рекомендаций

- `energy` и `valence` хорошо подходят для базовой карты настроений.
- `danceability` и `acousticness` помогают выделять dance/chill сценарии.
- Для рекомендаций похожих треков полезно использовать не один признак, а вектор audio features.
- Для похожих артистов данные агрегируются по артисту: средние energy, danceability, valence, tempo, acousticness, popularity.
- Жанр и popularity полезны как дополнительные фильтры для пользовательских плейлистов.